In [ ]:
import numpy as np
import galsim
from lsst.daf.butler import Butler
from lsst.summit.utils import ConsDbClient
from astropy.table import Table, QTable

In [ ]:
outdir = '~/u/rubin_aos/dk_to_dof_results'

In [ ]:
def get_visit_ids_from_consdb(visits_df, programs, img_type="cwfs", verify_pairing=True):
    """
    Extract visit information from ConsDB dataframe for FAM observations.

    Returns tuples of:
    (visit_id, day_obs, seq_num)

    For paired FAM cwfs images, only the second image in each consecutive pair
    is kept when verify_pairing=True.
    """
    program_mask = visits_df["science_program"].str.contains(programs[0], na=False)
    for prog in programs[1:]:
        program_mask |= visits_df["science_program"].str.contains(prog, na=False)

    filtered = visits_df[program_mask & (visits_df["img_type"] == img_type)].copy()

    if len(filtered) == 0:
        print("Warning: No matching visits found in ConsDB")
        return []

    filtered["visit_id"] = filtered.apply(
        lambda row: int(f"{int(row['day_obs'])}{int(row['seq_num']):05d}"),
        axis=1,
    )

    filtered = filtered.sort_values(["day_obs", "seq_num"])

    if not verify_pairing:
        visits = list(
            zip(
                filtered["visit_id"].astype(int),
                filtered["day_obs"].astype(int),
                filtered["seq_num"].astype(int),
            )
        )
        print(f"\nFound {len(visits)} matching visits (no pairing verification)")
        return visits

    visits = []
    unpaired_count = 0

    for day_obs, group in filtered.groupby("day_obs"):
        group = group.sort_values("seq_num").reset_index(drop=True)
        seq_nums = group["seq_num"].values

        i = 0
        while i < len(seq_nums) - 1:
            if seq_nums[i + 1] == seq_nums[i] + 1:
                row = group.iloc[i + 1]
                visits.append(
                    (
                        int(row["visit_id"]),
                        int(row["day_obs"]),
                        int(row["seq_num"]),
                    )
                )
                i += 2
            else:
                if unpaired_count < 5:
                    print(f"Warning: Unpaired image at day_obs={day_obs}, seq_num={seq_nums[i]}")
                unpaired_count += 1
                i += 1

        if i == len(seq_nums) - 1:
            if unpaired_count < 5:
                print(f"Warning: Unpaired image at day_obs={day_obs}, seq_num={seq_nums[i]}")
            unpaired_count += 1

    if unpaired_count > 5:
        print(f"... and {unpaired_count - 5} more unpaired images")

    print(f"\nFound {len(visits)} FAM image pairs (keeping second of each pair)")
    if unpaired_count > 0:
        print(f"Note: {unpaired_count} unpaired images were skipped")
        print("Tip: Set verify_pairing=False to include all images without pair checking")

    return visits

In [ ]:
fam_collections = [
    'aos_fam_danish_triplets'
]
butler = Butler('embargo', collections=fam_collections, instrument='LSSTCam')
os.environ["no_proxy"] += ",.consdb"
URL = "http://consdb-pq.consdb:8080/consdb"  
cdb_client = ConsDbClient(URL)

In [ ]:
day_obs = 20260315

day_obs_min = day_obs
day_obs_max = day_obs

# Query ConsDB for visit information
instrument = 'lsstcam'
visits_query = f'''
    SELECT 
        * 
    FROM 
        cdb_{instrument}.visit1 
    WHERE 
        day_obs >= {day_obs_min} AND day_obs <= {day_obs_max}
'''
visits = cdb_client.query(visits_query).to_pandas()
fam_programs = ['T278', 'T381', 'T492', 'T539', 'T614']
visit_pairs = get_visit_ids_from_consdb(visits, fam_programs, img_type='cwfs')


In [ ]:
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LinearRegression

per_visit_data = {}

for visit_name in tqdm(visit_pairs, total=len(visit_pairs)):
    visit_id = visit_name[0]

    aos_data = butler.get(
        "aggregateAOSVisitTableRaw",
        dataId={"instrument": "LSSTCam", "visit": visit_id},
    )

    thx = []
    thy = []
    zks = []

    for row in aos_data:
        thx.append(np.rad2deg(row["thx_OCS"]))  # degrees
        thy.append(np.rad2deg(row["thy_OCS"]))  # degrees
        zks.append(np.asarray(row["zk_OCS"], dtype=float))

    thx = np.asarray(thx, dtype=float)
    thy = np.asarray(thy, dtype=float)
    zks = np.asarray(zks, dtype=float)   # shape (n_points, n_zernikes)

    # Design matrix for linear plane: a*x + b*y + c
    X = np.column_stack([thx, thy])

    zks_linear_fit = np.zeros_like(zks)
    zks_corrected = np.zeros_like(zks)

    for i in range(zks.shape[1]):
        model = LinearRegression().fit(X, zks[:, i])
        z_fit = model.predict(X)

        zks_linear_fit[:, i] = z_fit
        zks_corrected[:, i] = zks[:, i] - z_fit

    per_visit_data[visit_id] = {
        "band": row.meta["band"],
        "thx": thx,
        "thy": thy,
        "zks_raw": zks,
        "zks_linear_fit": zks_linear_fit,
        "zks_corrected": zks_corrected,
    }

In [ ]:
bin2_dir = '/home/r/roodman/u/LSST/notebooks/rubin-work/aos/output/fam_danish_OCS_wep_v16_8_0_dviz_v3_5_0_20260315_20260317/'
fn = 'intrinsic_fits_wep_v16_8_0_dviz_v3_5_0_20260315_20260317.parquet'

In [ ]:
tab = QTable.read(bin2_dir + fn)

In [ ]:
tab

In [ ]:
tab['z1toz3_z4_c1']

In [ ]:
tab[tab['day_obs'] == 20260315]

In [ ]:
import numpy as np
import galsim

kMax = 15
jMin = 4
jMax = 22
fieldRadius = 1.75   # degrees
pupilOuter = 4.18
pupilInner = 2.558

all_coeffs = []

for visit_id, d in per_visit_data.items():
    x_vals = d["thx"]
    y_vals = d["thy"]
    zks = d["zks_corrected"][:, :23-4]   # Z4..Z12

    fieldAngles = np.column_stack([x_vals, y_vals])

    basis = galsim.zernike.zernikeBasis(
        kMax,
        fieldAngles[:, 0],
        fieldAngles[:, 1],
        R_outer=fieldRadius,
    )

    zernikesPadded = np.zeros((zks.shape[0], jMax + 1), dtype=float)
    zernikesPadded[:, jMin:jMax+1] = zks

    coeffs, *_ = np.linalg.lstsq(basis.T, zernikesPadded, rcond=None)
    coeffs[0, :] = 0.0
    coeffs[:, :jMin] = 0.0

    all_coeffs.append(coeffs)

all_coeffs = np.stack(all_coeffs, axis=0)
median_coeffs = np.median(all_coeffs, axis=0)

median_doubleZernikes = galsim.zernike.DoubleZernike(
    median_coeffs,
    uv_inner=0.0,
    uv_outer=fieldRadius,
    xy_inner=pupilInner,
    xy_outer=pupilOuter,
)

In [ ]:
np.max(np.rad2deg(d["thx"]))

In [ ]:

all_thx = []
all_thy = []
all_zks_corr = []

for visit_id, d in per_visit_data.items():
    all_thx.append(d["thx"])
    all_thy.append(d["thy"])
    all_zks_corr.append(d["zks_corrected"])

all_thx = np.concatenate(all_thx)
all_thy = np.concatenate(all_thy)
all_zks_corr = np.concatenate(all_zks_corr, axis=0)   # shape (Ntotal, n_zernikes)



In [ ]:
r_intrinsics = galsim.zernike.DoubleZernike(
    ofc_data.intrinsic_zk['R'],
    # Rubin annuli
    uv_inner=ofc_data.config["field"]["radius_inner"],
    uv_outer=ofc_data.config["field"]["radius_outer"],
    xy_inner=ofc_data.config["pupil"]["radius_inner"],
    xy_outer=ofc_data.config["pupil"]["radius_outer"],
)*0.622

all_zks_intrinsics = np.array([
    r_intrinsics(thx, thy).coef[jMin:jMax+1]
    for thx, thy in zip(all_thx[:30000], all_thy[:30000])
])   # shape (Ntotal, 9)

In [ ]:
jMin = 4
jMax = 22

all_zks_fit = np.array([
    median_doubleZernikes(thx, thy).coef[jMin:jMax+1]
    for thx, thy in zip(all_thx[:30000], all_thy[:30000])
])   # shape (Ntotal, 9)

In [ ]:
all_zks_fit.shape

In [ ]:
import matplotlib.pyplot as plt

def plot_hexbin_dz_comparison(
    thx,
    thy,
    zks_meas,
    zks_fit,
    zernikes_to_plot=range(4, 15),
    gridsize=25,
    saveAs='/home/g/gmegias/reference_wavefront/dz_fit.png',
    label='DZ fit'
):
    """
    Plot hexbin median maps of measured, fitted, and residual Zernikes.

    Parameters
    ----------
    thx, thy : array
        Field coordinates in degrees.
    zks_meas : array, shape (N, Nz)
        Measured corrected Zernikes, corresponding to zernikes_to_plot.
    zks_fit : array, shape (N, Nz)
        Double-Zernike fit evaluated at the same locations.
    zernikes_to_plot : iterable
        Actual Zernike numbers, e.g. range(4, 13).
    gridsize : int
        Hexbin resolution.
    """
    zernikes_to_plot = list(zernikes_to_plot)
    n_rows = len(zernikes_to_plot)

    fig, axs = plt.subplots(
        n_rows, 3, figsize=(16, 4 * n_rows), sharex=True, sharey=True
    )
    axs = np.atleast_2d(axs)

    for i, j in enumerate(zernikes_to_plot):
        idx = j - zernikes_to_plot[0]

        meas = zks_meas[:, idx]
        fit = zks_fit[:, idx]
        res = meas - fit

        # Use same color scale for measured and fit
        vmin = np.nanpercentile(np.concatenate([meas, fit]), 5)
        vmax = np.nanpercentile(np.concatenate([meas, fit]), 95)
        vmax = np.max([np.abs(vmin), vmax])
        vmin = -vmax

        hb1 = axs[i, 0].hexbin(
            thy, thx, C=meas,
            reduce_C_function=np.median,
            gridsize=gridsize,
            mincnt=1,
            vmin=vmin,
            vmax=vmax,
            cmap='seismic'
        )
        axs[i, 0].set_title(f"Median measured Z{j}")
        plt.colorbar(hb1, ax=axs[i, 0])

        hb2 = axs[i, 1].hexbin(
            thy, thx, C=fit,
            reduce_C_function=np.median,
            gridsize=gridsize,
            mincnt=1,
            vmin=vmin,
            vmax=vmax,
            cmap='seismic'
        )
        axs[i, 1].set_title(f"{label} Z{j}")
        plt.colorbar(hb2, ax=axs[i, 1])

        # Symmetric scale for residuals
        rlim = np.nanpercentile(np.abs(res), 95)

        hb3 = axs[i, 2].hexbin(
            thy, thx, C=res,
            reduce_C_function=np.median,
            gridsize=gridsize,
            mincnt=1,
            vmin=vmin,
            vmax=vmax,
            cmap='seismic'
        )
        axs[i, 2].set_title(f"Residual Z{j}")
        plt.colorbar(hb3, ax=axs[i, 2])

        for k in range(3):
            axs[i, k].set_xlabel("thx [deg]")
            axs[i, k].set_ylabel("thy [deg]")

    plt.tight_layout()
    plt.savefig(saveAs, dpi=300)

In [ ]:
x_hat_selected = np.zeros(50)
x_hat_selected[[21, 25, 29, 39]] = x_hat[[21, 25, 29, 39]]

In [ ]:
plot_hexbin_dz_comparison(
    all_thx[:30000],
    all_thy[:30000],
    all_zks_corr[:30000, :16],
    all_zks_fit,
    zernikes_to_plot=range(4, 16),
    gridsize=25,
)


In [ ]:
pwd

In [ ]:
from lsst.ts.ofc import OFCData, SensitivityMatrix
import galsim
ofc_data = OFCData('lsst')


In [ ]:
ideal_sens_dz = galsim.zernike.DoubleZernike(
    ofc_data.sensitivity_matrix[..., :],
    # Rubin annuli
    uv_inner=ofc_data.config["field"]["radius_inner"],
    uv_outer=ofc_data.config["field"]["radius_outer"],
    xy_inner=ofc_data.config["pupil"]["radius_inner"],
    xy_outer=ofc_data.config["pupil"]["radius_outer"],
)

In [ ]:
median_doubleZernikes.coef[:3, 4:23].shape

In [ ]:
ideal_sens_dz.coef[:3, 4:23, :].shape

In [ ]:
# Measured DZ coefficients
y_dz = median_doubleZernikes.coef[:10, 4:15] - r_intrinsics.coef[:10, 4:15]

# Sensitivity tensor
S = ideal_sens_dz.coef[:10, 4:15, :]  # shape (16, 23, 50)

# Flatten
y = y_dz.reshape(-1)                   # (368,)
A = S.reshape(-1, 50)                  # (368, 50)

# Solve
x_hat, residuals, rank, s = np.linalg.lstsq(A, y, rcond=1e-3)

print("Recovered DOFs shape:", x_hat.shape)
print("Rank:", rank)
print("Singular values:", s)

In [ ]:
(ideal_sens_dz.coef[:10, 4:15, :]@x_hat).shape

In [ ]:
x_hat_selected = np.zeros(50)
x_hat_selected[[21, 25, 29, 39]] = x_hat[[21, 25, 29, 39]]

In [ ]:
x_contribution = galsim.zernike.DoubleZernike(
    ideal_sens_dz.coef[:10, 4:23, :]@x_hat_selected,
    # Rubin annuli
    uv_inner=ofc_data.config["field"]["radius_inner"],
    uv_outer=ofc_data.config["field"]["radius_outer"],
    xy_inner=ofc_data.config["pupil"]["radius_inner"],
    xy_outer=ofc_data.config["pupil"]["radius_outer"],
)

all_zks_contribution = np.array([
    x_contribution(thx, thy).coef[jMin:jMax+1]
    for thx, thy in zip(all_thx[:30000], all_thy[:30000])
])   # shape (Ntotal, 9)

In [ ]:
all_zks_contribution.shape

In [ ]:
plot_hexbin_dz_comparison(
    all_thx[:30000],
    all_thy[:30000],
    all_zks_corr[:30000, :16],
    all_zks_contribution,
    zernikes_to_plot=range(4, 16),
    gridsize=25,
    saveAs=f'{output_dir}/{day_obs}_measured_minus_state_correction_selected.png',
    label='optical_state (only m1m3 b16,b12,b20 and m2b20)'
)

In [ ]:
dof_labels = [ "M2_hex_z", "M2_hex_x", "M2_hex_y", "M2_hex_rx", "M2_hex_ry", "Cam_hex_z", "Cam_hex_x", "Cam_hex_y", "Cam_hex_rx", "Cam_hex_ry", ] 
dof_labels += [f"M1M3_B{i}" for i in range(1, 21)] 
dof_labels += [f"M2_B{i}" for i in range(1, 21)]

In [ ]:
import numpy as np

x_hat = np.asarray(x_hat)
x_hat = np.asarray(x_hat_selected)
label_to_value = dict(zip(dof_labels, x_hat))

left_groups = [
    ("Camera hexapod", dof_labels[5:10]),
    ("M1M3 bends", dof_labels[10:30]),
]

right_groups = [
    ("M2 hexapod", dof_labels[:5]),
    ("M2 bends", dof_labels[30:50]),
]

def build_rows(group_specs):
    out = []
    for group_name, labels in group_specs:
        rows = []
        for label in labels:
            val = label_to_value[label]
            if label.endswith("_rx") or label.endswith("_ry"):
                display_val = val * 3600.0
                unit = "arcsec"
            else:
                display_val = val
                unit = "um"
            rows.append((label, display_val, unit))
        out.append((group_name, rows))
    return out

left_data = build_rows(left_groups)
right_data = build_rows(right_groups)

all_rows = [row for _, rows in (left_data + right_data) for row in rows]
label_w = max(len("DOF"), max(len(r[0]) for r in all_rows))
value_w = max(len("Value"), 14)
unit_w = max(len("Unit"), max(len(r[2]) for r in all_rows))

def make_block(group_name, rows):
    line = f"+-{'-'*label_w}-+-{'-'*value_w}-+-{'-'*unit_w}-+"
    group_line_w = len(line) - 4

    block = []
    block.append(line)
    block.append(f"| {'DOF':<{label_w}} | {'Value':>{value_w}} | {'Unit':<{unit_w}} |")
    block.append(line)
    block.append(f"| {(' ' + group_name + ' '):<{group_line_w}} |")
    block.append(line)
    for label, value, unit in rows:
        block.append(f"| {label:<{label_w}} | {value:>{value_w}.6f} | {unit:<{unit_w}} |")
    block.append(line)
    return block

def flatten_blocks(grouped_blocks):
    lines = []
    for i, (group_name, rows) in enumerate(grouped_blocks):
        lines.extend(make_block(group_name, rows))
        if i != len(grouped_blocks) - 1:
            lines.append("")
    return lines

left_lines = flatten_blocks(left_data)
right_lines = flatten_blocks(right_data)

left_width = max(len(line) for line in left_lines)
right_width = max(len(line) for line in right_lines)
n_lines = max(len(left_lines), len(right_lines))

left_lines += [""] * (n_lines - len(left_lines))
right_lines += [""] * (n_lines - len(right_lines))

gap = "   "

for l, r in zip(left_lines, right_lines):
    print(f"{l:<{left_width}}{gap}{r}")

In [ ]:
x_hat_selected = np.zeros(50)
x_hat_selected[[21, 25, 29, 39]] = x_hat[[21, 25, 29, 39]]

In [ ]:


# Inputs
# x_vals, y_vals in degrees
# zks shape = (num_points, 9), corresponding to pupil j = 4..12

x_vals = np.asarray(x_vals, dtype=float)
y_vals = np.asarray(y_vals, dtype=float)
zks = np.asarray(zks, dtype=float)

fieldAngles = np.column_stack([x_vals, y_vals])

# Double-Zernike settings
kMax = 15              # field Zernike max
jMin = 4
jMax = 12
fieldRadius = 1.75     # degrees
pupilOuter = 4.18
pupilInner = 2.558

# Build field Zernike basis
# basis has shape (kMax+1, num_points)
basis = galsim.zernike.zernikeBasis(
    kMax,
    fieldAngles[:, 0],
    fieldAngles[:, 1],
    R_outer=fieldRadius,
)

# Pad measured pupil Zernikes into full j-indexed array
# shape will be (num_points, jMax+1)
zernikesPadded = np.zeros((zks.shape[0], jMax + 1), dtype=float)
zernikesPadded[:, jMin:jMax+1] = zks

# Solve basis.T @ coeffs = zernikesPadded
# coeffs shape = (kMax+1, jMax+1)
doubleZernikeCoeffs, *_ = np.linalg.lstsq(
    basis.T,
    zernikesPadded,
    rcond=None,
)

# Zero meaningless / unwanted terms
doubleZernikeCoeffs[0, :] = 0.0      # k=0 field piston-like term
doubleZernikeCoeffs[:, :jMin] = 0.0  # ignore pupil j < 4

# Build DoubleZernike object
doubleZernikes = galsim.zernike.DoubleZernike(
    doubleZernikeCoeffs,
    uv_inner=0.0,
    uv_outer=fieldRadius,
    xy_inner=pupilInner,
    xy_outer=pupilOuter,
)